# 04. Decomposed SIR Evaluation (Symbolic)

**Purpose:** Compute the decomposed Semantic Irrelevance Ratio (SIR*) over the full KG path set and the symbolically filtered path set.

**What SIR measures (§3.4):** At each decoding step $t$, what fraction of paths in the valid token set are semantically irrelevant? Irrelevance is the disjunction of two independent failure modes:
- **Type irrelevance** ($\text{SIR}^*_{\text{type}}$): terminal entity type mismatches the question's answer type
- **Trajectory irrelevance** ($\text{SIR}^*_{\text{traj}}$): a relation in the path has a declared range incompatible with the question's answer type

**Reference:** Chapter 3, §3.4. Equations (3.5)-(3.11).

Key formulas:
```
irrel_type(p, q) = 1[type(e_h) ∉ T(q, h)]                          (Eq. 3.5)
irrel_traj(p, q) = 1[∃ hop i: range(r_i) ∩ T(q, h) = ∅]           (Eq. 3.6)
irrel*(p, q) = irrel_type ∨ irrel_traj                              (Eq. 3.7)
SIR*_type(q,t) = count(irrel_type=1) / |P(q,t)|                    (Eq. 3.8)
SIR*_traj(q,t) = count(irrel_traj=1) / |P(q,t)|                    (Eq. 3.9)
SIR*(q,t) = count(irrel*=1) / |P(q,t)|                             (Eq. 3.10)
```

In [ ]:
# @title 1. Setup
import sys, os, json, re
from collections import defaultdict
from tqdm import tqdm
import numpy as np

!pip install -q datasets marisa-trie networkx

if not os.path.exists("graph-constrained-reasoning"):
    !git clone https://github.com/rmanluo/graph-constrained-reasoning.git
%cd graph-constrained-reasoning
sys.path.insert(0, '.')

from datasets import load_dataset
import src.utils as utils

# Import symbolic TypeOracle
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from type_oracle import TypeOracle, compute_type_irrelevance

In [ ]:
# @title 2. Configuration
# Dataset
DATA_PATH = "rmanluo"
DATASET = "RoG-webqsp"
SPLIT = "test"
INDEX_LEN = 2

# ═══════════════════════════════════════════════════
# SAMPLING
# ═══════════════════════════════════════════════════
MAX_SAMPLES = 100          # set to None for full dataset

# No encoder, no τ_ref. All SIR metrics are symbolic.

In [ ]:
# @title 3. Symbolic SIR Computation (§3.4.2–3.4.3)

def compute_symbolic_sir(paths_list, oracle, answer_types, max_hop):
    """
    Compute decomposed SIR metrics for a single path set.

    Returns per-question dict with:
      - sir_type: fraction of paths with type irrelevance (Eq. 3.8)
      - sir_traj: fraction of paths with trajectory irrelevance (Eq. 3.9)
      - sir: fraction of paths with either (Eq. 3.10)
      - Per-hop breakdowns
    """
    paths_by_hop = defaultdict(list)
    for p in paths_list:
        paths_by_hop[len(p)].append(p)

    hop_sir_type = {}
    hop_sir_traj = {}
    hop_sir = {}
    hop_counts = {}

    for step, hop_paths in paths_by_hop.items():
        n_total = len(hop_paths)
        hop_counts[step] = n_total

        n_type_irr = 0
        n_traj_irr = 0
        n_combined = 0

        for p in hop_paths:
            terminal_entity = p[-1][2]
            h = len(p)

            # ── TYPE IRRELEVANCE (Eq. 3.5) ──
            type_bad = not oracle.type_gate(
                terminal_entity, answer_types, h, max_hop
            )

            # ── TRAJECTORY IRRELEVANCE (Eq. 3.6) ──
            # Does any hop use a relation whose range doesn't intersect
            # the question's answer type?
            traj_bad = False
            for _, relation, _ in p:
                rel_schema = oracle._schema.get(relation)
                if rel_schema is None:
                    continue
                range_types = rel_schema.get("range", frozenset())
                if not range_types:
                    continue
                if not (range_types & answer_types):
                    traj_bad = True
                    break

            if type_bad:
                n_type_irr += 1
            if traj_bad:
                n_traj_irr += 1
            if type_bad or traj_bad:
                n_combined += 1

        hop_sir_type[step] = n_type_irr / max(1, n_total)
        hop_sir_traj[step] = n_traj_irr / max(1, n_total)
        hop_sir[step] = n_combined / max(1, n_total)

    return {
        "hop_sir_type": hop_sir_type,
        "hop_sir_traj": hop_sir_traj,
        "hop_sir": hop_sir,
        "hop_counts": hop_counts,
    }

In [ ]:
# @title 4. Compute SIR for Full KG Paths (GCR Baseline)
print("=" * 70)
print("SIR OVER FULL KG PATH SET (what GCR sees)")
print("=" * 70)

dataset = load_dataset(f"{DATA_PATH}/{DATASET}", split=SPLIT)
if MAX_SAMPLES is not None:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))
    print(f"Subsampled to {len(dataset)} questions")

# Accumulators
all_sir_type = []
all_sir_traj = []
all_sir = []
hop_sir_type_agg = defaultdict(list)
hop_sir_traj_agg = defaultdict(list)
hop_sir_agg = defaultdict(list)
hop_count_agg = defaultdict(list)

for data in tqdm(dataset, desc="Full KG SIR"):
    question = data["question"]
    entities = data["q_entity"] if "q_entity" in data else []

    # Build oracle
    oracle = TypeOracle.from_graph(data["graph"])
    answer_types = oracle.infer_answer_types(question)

    # Get all BFS paths
    if "paths" in data:
        paths_list = data["paths"]
    else:
        g = utils.build_graph(data["graph"])
        paths_list = utils.dfs(g, entities, INDEX_LEN)

    if len(paths_list) == 0:
        continue

    result = compute_symbolic_sir(paths_list, oracle, answer_types, INDEX_LEN)

    for step in result["hop_sir_type"]:
        hop_sir_type_agg[step].append(result["hop_sir_type"][step])
        hop_sir_traj_agg[step].append(result["hop_sir_traj"][step])
        hop_sir_agg[step].append(result["hop_sir"][step])
        hop_count_agg[step].append(result["hop_counts"][step])
        all_sir_type.append(result["hop_sir_type"][step])
        all_sir_traj.append(result["hop_sir_traj"][step])
        all_sir.append(result["hop_sir"][step])

print(f"\nCorpus-level SIR (GCR baseline, {len(all_sir)} observations):")
print(f"  SIR*_type: {np.mean(all_sir_type):.4f}")
print(f"  SIR*_traj: {np.mean(all_sir_traj):.4f}")
print(f"  SIR*:      {np.mean(all_sir):.4f}")

print("\nPer-hop SIR:")
for step in sorted(hop_sir_agg.keys()):
    print(f"  Hop-{step} | n={len(hop_sir_agg[step]):4d} | "
          f"SIR*={np.mean(hop_sir_agg[step]):.4f} | "
          f"SIR*_type={np.mean(hop_sir_type_agg[step]):.4f} | "
          f"SIR*_traj={np.mean(hop_sir_traj_agg[step]):.4f} | "
          f"Avg paths={np.mean(hop_count_agg[step]):.0f}")

In [ ]:
# @title 5. Compute SIR for DCA-Trie v1 (Symbolically Filtered Paths)
print("=" * 70)
print("SIR OVER DCA-TRIE v1 FILTERED PATHS")
print("=" * 70)

dataset_v1 = load_dataset(f"{DATA_PATH}/{DATASET}", split=SPLIT)
if MAX_SAMPLES is not None:
    dataset_v1 = dataset_v1.select(range(min(MAX_SAMPLES, len(dataset_v1))))

all_sir_type_v1 = []
all_sir_traj_v1 = []
all_sir_v1 = []
hop_sir_type_v1 = defaultdict(list)
hop_sir_traj_v1 = defaultdict(list)
hop_sir_v1 = defaultdict(list)
hop_count_v1 = defaultdict(list)
n_empty_v1 = 0

for data in tqdm(dataset_v1, desc="DCA-v1 SIR"):
    question = data["question"]
    entities = data["q_entity"] if "q_entity" in data else []

    # Build oracle and filter paths through both gates
    oracle = TypeOracle.from_graph(data["graph"])
    answer_types = oracle.infer_answer_types(question)

    # Get all BFS paths
    if "paths" in data:
        paths_list = data["paths"]
    else:
        g = utils.build_graph(data["graph"])
        paths_list = utils.dfs(g, entities, INDEX_LEN)

    if len(paths_list) == 0:
        continue

    # Apply symbolic gates to filter (same as Algorithm 1)
    filtered_paths = []
    for p in paths_list:
        h = len(p)
        admit = True

        # Range gate at every hop
        for _, relation, tail_entity in p:
            if not oracle.range_gate(relation, tail_entity):
                admit = False
                break

        # Type gate at terminal hop
        if admit:
            terminal_entity = p[-1][2]
            if not oracle.type_gate(terminal_entity, answer_types, h, INDEX_LEN):
                admit = False

        if admit:
            filtered_paths.append(p)

    if len(filtered_paths) == 0:
        n_empty_v1 += 1
        continue

    result = compute_symbolic_sir(filtered_paths, oracle, answer_types, INDEX_LEN)

    for step in result["hop_sir_type"]:
        hop_sir_type_v1[step].append(result["hop_sir_type"][step])
        hop_sir_traj_v1[step].append(result["hop_sir_traj"][step])
        hop_sir_v1[step].append(result["hop_sir"][step])
        hop_count_v1[step].append(result["hop_counts"][step])
        all_sir_type_v1.append(result["hop_sir_type"][step])
        all_sir_traj_v1.append(result["hop_sir_traj"][step])
        all_sir_v1.append(result["hop_sir"][step])

print(f"\nFiltered to empty trie: {n_empty_v1} questions")
print(f"Corpus-level SIR (DCA-v1, {len(all_sir_v1)} observations):")
print(f"  SIR*_type: {np.mean(all_sir_type_v1):.4f}")
print(f"  SIR*_traj: {np.mean(all_sir_traj_v1):.4f}")
print(f"  SIR*:      {np.mean(all_sir_v1):.4f}")

print("\nPer-hop SIR:")
for step in sorted(hop_sir_v1.keys()):
    print(f"  Hop-{step} | n={len(hop_sir_v1[step]):4d} | "
          f"SIR*={np.mean(hop_sir_v1[step]):.4f} | "
          f"SIR*_type={np.mean(hop_sir_type_v1[step]):.4f} | "
          f"SIR*_traj={np.mean(hop_sir_traj_v1[step]):.4f} | "
          f"Avg paths={np.mean(hop_count_v1[step]):.0f}")

In [ ]:
# @title 6. Comparison: GCR vs DCA-Trie v1 SIR Reduction
print("=" * 70)
print("SIR REDUCTION: DCA-Trie v1 vs GCR")
print("=" * 70)

if len(all_sir) > 0 and len(all_sir_v1) > 0:
    def reduction(gcr, dca):
        return (gcr - dca) / max(gcr, 1e-8) * 100

    print(f"\n{'Metric':<20} {'GCR':<10} {'DCA-v1':<10} {'Reduction':<10}")
    print("-" * 50)
    print(f"{'SIR*':<20} {np.mean(all_sir):<10.4f} {np.mean(all_sir_v1):<10.4f} "
          f"{reduction(np.mean(all_sir), np.mean(all_sir_v1)):<10.1f}%")
    print(f"{'SIR*_type':<20} {np.mean(all_sir_type):<10.4f} {np.mean(all_sir_type_v1):<10.4f} "
          f"{reduction(np.mean(all_sir_type), np.mean(all_sir_type_v1)):<10.1f}%")
    print(f"{'SIR*_traj':<20} {np.mean(all_sir_traj):<10.4f} {np.mean(all_sir_traj_v1):<10.4f} "
          f"{reduction(np.mean(all_sir_traj), np.mean(all_sir_traj_v1)):<10.1f}%")

    print("\nPer-hop comparison:")
    for step in sorted(set(list(hop_sir_agg.keys()) + list(hop_sir_v1.keys()))):
        gcr_s = np.mean(hop_sir_agg.get(step, [0]))
        v1_s = np.mean(hop_sir_v1.get(step, [0]))
        gcr_st = np.mean(hop_sir_type_agg.get(step, [0]))
        v1_st = np.mean(hop_sir_type_v1.get(step, [0]))
        gcr_str = np.mean(hop_sir_traj_agg.get(step, [0]))
        v1_str = np.mean(hop_sir_traj_v1.get(step, [0]))

        print(f"\n  Hop-{step}:")
        print(f"    SIR*:      {gcr_s:.4f} -> {v1_s:.4f} ({reduction(gcr_s, v1_s):.1f}%)")
        print(f"    SIR*_type: {gcr_st:.4f} -> {v1_st:.4f} ({reduction(gcr_st, v1_st):.1f}%)")
        print(f"    SIR*_traj: {gcr_str:.4f} -> {v1_str:.4f} ({reduction(gcr_str, v1_str):.1f}%)")

In [ ]:
# @title 7. Per-Gate Irrelevance Attribution
print("=" * 70)
print("IRRELEVANCE ATTRIBUTION BY GATE")
print("=" * 70)
print("""
SIR*_type  = fraction of paths where terminal entity type != answer type
             → these are paths the type gate would prune

SIR*_traj  = fraction of paths where a relation's range != answer type
             → these are paths the range gate would prune

If SIR*_type = 0.6 and SIR*_traj = 0.7, that means:
  - 60% of paths have the wrong terminal type
  - 70% have at least one relation with incompatible range
  - The combined SIR* is NOT 1.3 because these overlap
  (Some paths have BOTH type and trajectory irrelevance.)

These two metrics directly measure the two design gaps from
Chapter 2: type blindness (SIR*_type) and trajectory blindness
(SIR*_traj). Each is actionable independently.
""")

# Additional: Venn overlap
print("--- Venn overlap (type-only vs traj-only vs both) ---")
dataset_overlap = load_dataset(f"{DATA_PATH}/{DATASET}", split=SPLIT)
if MAX_SAMPLES is not None:
    dataset_overlap = dataset_overlap.select(range(min(MAX_SAMPLES, len(dataset_overlap))))

total_paths = 0
n_type_only = 0
n_traj_only = 0
n_both = 0
n_neither = 0

for data in tqdm(dataset_overlap, desc="Overlap analysis"):
    entities = data["q_entity"] if "q_entity" in data else []
    oracle = TypeOracle.from_graph(data["graph"])
    answer_types = oracle.infer_answer_types(data["question"])

    if "paths" in data:
        paths_list = data["paths"]
    else:
        g = utils.build_graph(data["graph"])
        paths_list = utils.dfs(g, entities, INDEX_LEN)

    for p in paths_list:
        h = len(p)
        total_paths += 1

        # Type irrelevance
        terminal_entity = p[-1][2]
        type_bad = not oracle.type_gate(terminal_entity, answer_types, h, INDEX_LEN)

        # Trajectory irrelevance
        traj_bad = False
        for _, relation, _ in p:
            rel_schema = oracle._schema.get(relation)
            if rel_schema is None:
                continue
            range_types = rel_schema.get("range", frozenset())
            if not range_types:
                continue
            if not (range_types & answer_types):
                traj_bad = True
                break

        if type_bad and not traj_bad:
            n_type_only += 1
        elif traj_bad and not type_bad:
            n_traj_only += 1
        elif type_bad and traj_bad:
            n_both += 1
        else:
            n_neither += 1

print(f"\nTotal paths analyzed: {total_paths}")
print(f"  Type-only irrelevant:  {n_type_only:6d} ({n_type_only/max(1,total_paths)*100:.1f}%)")
print(f"  Traj-only irrelevant:  {n_traj_only:6d} ({n_traj_only/max(1,total_paths)*100:.1f}%)")
print(f"  Both irrelevant:       {n_both:6d} ({n_both/max(1,total_paths)*100:.1f}%)")
print(f"  Neither (relevant):    {n_neither:6d} ({n_neither/max(1,total_paths)*100:.1f}%)")
print()
print(f"  Total irrelevant:      {n_type_only + n_traj_only + n_both:6d} "
      f"({(n_type_only + n_traj_only + n_both)/max(1,total_paths)*100:.1f}%)")
print(f"  Total relevant:        {n_neither:6d} ({n_neither/max(1,total_paths)*100:.1f}%)")

In [ ]:
# @title 8. Interpretation
print("""
=== INTERPRETATION (§3.4.4) ===

SIR*_type isolates the contribution of type blindness.
  - GCR: high (no type filtering)
  - DCA-Trie v1: ~0 (type gate kills these)

SIR*_traj isolates trajectory blindness.
  - GCR: high (no range filtering) 
  - DCA-Trie v1: low (range gate kills these)

The Venn overlap analysis shows whether type and trajectory
irrelevance are independent failure modes or correlate.

Both metrics are purely symbolic — no embeddings, no thresholds.
They respond directly to Gap 3 in Chapter 2 by attributing
the contribution of each filtering mechanism independently.
""")